# RTL Gate Count Estimator — Analysis Notebook

Load pipeline outputs and visualize model performance, SHAP explanations, ablation results, and faithfulness metrics.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('..').resolve()
RESULTS = PROJECT_ROOT / 'results'
DATA = PROJECT_ROOT / 'data'

sns.set_theme(style='whitegrid')
print(f'Project root: {PROJECT_ROOT}')

## 1. Dataset Overview

In [ ]:
features = pd.read_csv(DATA / 'features' / 'features.csv')
labels = pd.read_csv(DATA / 'labels' / 'labels.csv')

print(f'Features: {features.shape}')
print(f'Labels: {labels.shape}')
display(features.head())
display(labels.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels['gate_count'].hist(ax=axes[0], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Gate Count Distribution')
axes[0].set_xlabel('Gate count')

feature_cols = [c for c in features.columns if c != 'module_id']
features[feature_cols].corrwith(labels['gate_count']).sort_values(ascending=False).head(10).plot(
    kind='barh', ax=axes[1], color='darkorange'
)
axes[1].set_title('Top Feature Correlations with Gate Count')
plt.tight_layout()
plt.show()

## 2. Training Metrics

In [ ]:
metrics_path = RESULTS / 'training_metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    for split, values in metrics['metrics'].items():
        print(f"{split.upper():5} R2={values['r2']:.4f}  MAE={values['mae']:.4f}  MSE={values['mse']:.4f}")

    top_features = metrics['feature_importance'][:10]
    fi_df = pd.DataFrame(top_features)
    fi_df.plot.bar(x='feature', y='importance', legend=False, figsize=(10, 4), color='seagreen')
    plt.title('Random Forest Feature Importance')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Run phase 4 first: python scripts/train_model.py ...')

## 3. SHAP Visualizations

In [ ]:
from IPython.display import Image, display

for plot_name in ['shap_summary.png', 'shap_beeswarm.png']:
    plot_path = RESULTS / plot_name
    if plot_path.exists():
        print(plot_name)
        display(Image(filename=str(plot_path)))
    else:
        print(f'Missing {plot_name}. Run: python scripts/run_shap.py ...')

## 4. Ablation vs SHAP Faithfulness

In [ ]:
faithfulness_path = RESULTS / 'faithfulness_report.json'
ablation_path = RESULTS / 'ablation_study_results.json'
shap_path = RESULTS / 'shap_values.json'

if faithfulness_path.exists():
    report = json.loads(faithfulness_path.read_text())
    print(f"Faithfulness score: {report['faithfulness_score']:.4f}")
    print(f"Spearman correlation: {report['correlation_metrics']['spearman_correlation']:.4f}")
    print(f"Interpretation: {report['interpretation']}")

if ablation_path.exists() and shap_path.exists():
    ablation = pd.DataFrame(json.loads(ablation_path.read_text()))
    shap_data = json.loads(shap_path.read_text())
    shap_imp = pd.DataFrame(shap_data['feature_importance'])

    merged = ablation.merge(shap_imp, on='feature', suffixes=('_ablation', '_shap'))
    merged.plot.scatter(
        x='ablation_score', y='importance', figsize=(7, 5), color='purple', alpha=0.8
    )
    for _, row in merged.iterrows():
        plt.annotate(row['feature'], (row['ablation_score'], row['importance']), fontsize=7)
    plt.title('Ablation vs SHAP Feature Importance')
    plt.xlabel('Ablation score')
    plt.ylabel('Mean |SHAP|')
    plt.tight_layout()
    plt.show()

## 5. Pipeline Summary

In [ ]:
summary_path = RESULTS / 'pipeline_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    for phase, info in summary['phases'].items():
        print(phase, info)
else:
    print('Run full pipeline: python run_pipeline.py')